In [ ]:
# =========================
# nb_ingest_wh_ireland
# =========================

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    # Standalone execution fallback
    pass

dfm_id = "wh_ireland"
dfm_name = "WH Ireland"

# ---------- Imports ----------
from pyspark.sql import functions as F, types as T
from datetime import datetime, timezone
import pandas as pd
import re

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
fx_path = "Files/config/fx_rates.csv"
print(f"[DEBUG] Landing path: {landing_path}")
print(f"[DEBUG] FX path: {fx_path}")

# ---------- Spec-ground-truth columns ----------
required_cols = [
    "Position as of Date",
    "Account Number",
    "Settled Quantity",
    "Display Price",
    "Settled Market Value (PC)",
    "Position Currency"
]
optional_cols = [
    "ISIN",
    "Position to Account Exchange Rate",
    "Settled Market Value (ABC)",
    "Account Base Currency"
]

# ---------- Helpers ----------
def q(s: str) -> str:
    """Escape single quotes for SQL literals."""
    return (s or "").replace("'", "''")

def to_norm(c: str) -> str:
    """Normalize source column names for DataFrame use."""
    return re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_")

def dec(col_name: str):
    """Cast column to DECIMAL(28, 10) for canonical holdings."""
    return F.col(col_name).cast("DECIMAL(28, 10)")

def write_audit(files_processed: int, rows_ingested: int, parse_errors_count: int, drift_events_count: int, status: str):
    """Write audit log entry."""
    print(f"[AUDIT] files_processed={files_processed}, rows_ingested={rows_ingested}, parse_errors={parse_errors_count}, drift_events={drift_events_count}, status={status}")
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{q(run_id)}', '{q(period)}', '{q(dfm_id)}', {int(files_processed)}, {int(rows_ingested)}, {int(parse_errors_count)}, {int(drift_events_count)}, '{q(status)}', current_timestamp(), current_timestamp())
    """)

def write_parse_errors(df_err):
    """Write parse errors to parse_errors table."""
    err_count = df_err.count()
    print(f"[DEBUG] Writing {err_count} parse errors to parse_errors table")
    cols = set(df_err.columns)
    out = df_err

    if "source_file" not in cols:
        out = out.withColumn("source_file", F.lit("UNKNOWN"))
    if "row_number" not in cols:
        if "source_row_id" in cols:
            out = out.withColumn("row_number", F.regexp_extract(F.col("source_row_id").cast("string"), r"(\\d+)$", 1).cast("bigint"))
        else:
            out = out.withColumn("row_number", F.lit(None).cast("bigint"))
    if "column_name" not in cols:
        out = out.withColumn("column_name", F.lit(None).cast("string"))
    if "raw_value" not in cols:
        out = out.withColumn("raw_value", F.lit(None).cast("string"))
    if "error_type" not in cols:
        out = out.withColumn("error_type", F.lit("PARSE_ERROR"))
    if "error_message" not in cols:
        out = out.withColumn("error_message", F.lit("Unknown parse error"))

    (
        out.withColumn("period", F.lit(period))
           .withColumn("run_id", F.lit(run_id))
           .withColumn("dfm_id", F.lit(dfm_id))
           .withColumn("event_ts", F.current_timestamp())
           .select("period","run_id","dfm_id","source_file","row_number","column_name","raw_value","error_type","error_message","event_ts")
           .write.mode("append").saveAsTable("parse_errors")
    )
print("[A3] Starting file discovery...")
try:
    all_entries = mssparkutils.fs.ls(landing_path)
    print(f"[A3] Found {len(all_entries)} entries in landing path")
except Exception:
    # path missing treated as NO_FILES
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")
xlsx_files = [f for f in all_entries if (not f.isDir and f.path.lower().endswith(".xlsx"))]

if len(xlsx_files) == 0:
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")
print(f"[A4] Starting file parsing... xlsx_files={len(xlsx_files)}")
# For WH Ireland spec, expect a single valuation workbook; handle multi by union.
pdfs = []
parse_err_rows = []

for f in xlsx_files:
    try:
        xls = pd.ExcelFile(f.path)
    except Exception as ex:
        parse_err_rows.append((f.name, "FILE", f"Unable to open workbook: {str(ex)}", ""))
        continue

    matched_sheet = None
    matched_pdf = None

    # Auto-detect sheet that contains required columns
    for sheet in xls.sheet_names:
        try:
            tmp = pd.read_excel(f.path, sheet_name=sheet)
            cols = [str(c).strip() for c in tmp.columns]
            if all(req in cols for req in required_cols):
                matched_sheet = sheet
                matched_pdf = tmp
                break
        except Exception:
            continue

    if matched_pdf is None:
        parse_err_rows.append((f.name, "SHEET", "No sheet contains all required WH Ireland columns; using raw preservation fallback", ""))
        for sheet in xls.sheet_names:
            try:
                raw_sheet = pd.read_excel(f.path, sheet_name=sheet, header=None, dtype=str)
                if len(raw_sheet) == 0:
                    continue
                fallback_pdf = pd.DataFrame()
                for c in required_cols + optional_cols:
                    fallback_pdf[c] = None
                fallback_pdf["raw_row_payload"] = raw_sheet.fillna("").astype(str).agg("|".join, axis=1)
                fallback_pdf["source_file"] = f.name
                fallback_pdf["source_sheet"] = sheet
                pdfs.append(fallback_pdf)
            except Exception:
                continue
        continue

    # Keep source metadata
    matched_pdf["source_file"] = f.name
    matched_pdf["source_sheet"] = matched_sheet
    pdfs.append(matched_pdf)

if len(parse_err_rows) > 0:
    df_err_file = spark.createDataFrame(parse_err_rows, ["source_file", "source_row_id", "error_message", "raw_value"])
    write_parse_errors(df_err_file)

if len(pdfs) == 0:
    # files existed but none parseable
    write_audit(len(xlsx_files), 0, len(parse_err_rows), 0, "FAILED")
    mssparkutils.notebook.exit("FAILED")

pdf_all = pd.concat(pdfs, ignore_index=True)

# ---------- Standardize raw columns ----------
rename_map = {c: to_norm(c) for c in pdf_all.columns}
pdf_all = pdf_all.rename(columns=rename_map)

# Build Spark DF
df_raw = spark.createDataFrame(pdf_all)

# Create deterministic source row id per source file/sheet
w = F.window
df_raw = df_raw.withColumn("_rownum", F.row_number().over(
    __import__("pyspark").sql.window.Window.partitionBy("source_file", "source_sheet").orderBy(F.monotonically_increasing_id())
))
df_raw = df_raw.withColumn("source_row_id", F.concat_ws(":", F.col("source_file"), F.col("source_sheet"), F.col("_rownum").cast("string")))
if "raw_row_payload" not in df_raw.columns:
    payload_cols = [c for c in df_raw.columns if c not in ["source_file", "source_sheet", "_rownum", "source_row_id"]]
    if len(payload_cols) > 0:
        df_raw = df_raw.withColumn("raw_row_payload", F.concat_ws("|", *[F.col(c).cast("string") for c in payload_cols]))
    else:
        df_raw = df_raw.withColumn("raw_row_payload", F.lit(None).cast("string"))

print("[A5/A6] Mapping to canonical schema + applying FX rates...")
# Source normalized names:
# position_as_of_date, account_number, settled_quantity, display_price,
# settled_market_value_pc, position_currency, isin (optional),
# position_to_account_exchange_rate (optional),
# settled_market_value_abc (optional),
# account_base_currency (optional)

# Add missing optional columns if absent
for c in ["isin", "position_to_account_exchange_rate", "settled_market_value_abc", "account_base_currency"]:
    if c not in df_raw.columns:
        df_raw = df_raw.withColumn(c, F.lit(None).cast("string"))

# Ensure required columns exist (after normalization)
norm_required = [to_norm(c) for c in required_cols]
missing_required = [c for c in norm_required if c not in df_raw.columns]

if missing_required:
    df_miss = spark.createDataFrame(
        [(f.name, "HEADER", f"Missing required columns after normalization; filled as NULL: {','.join(missing_required)}", "") for f in xlsx_files],
        ["source_file", "source_row_id", "error_message", "raw_value"]
    )
    write_parse_errors(df_miss)
    for mc in missing_required:
        df_raw = df_raw.withColumn(mc, F.lit(None).cast("string"))

# FX table
fx_df = (
    spark.read.option("header", True).csv(fx_path)
    .withColumn("currency_u", F.upper(F.trim(F.col("currency"))))
    .withColumn("rate_to_gbp_d", F.col("rate_to_gbp").cast("double"))
    .select("currency_u", "rate_to_gbp_d")
)

d = (
    df_raw
    .withColumn("policy_id", F.col("account_number").cast("string"))
    .withColumn("dfm_policy_id", F.col("account_number").cast("string"))
    .withColumn("isin_out", F.col("isin").cast("string"))
    .withColumn("security_id", F.col("isin").cast("string"))
    .withColumn("holding_d", F.col("settled_quantity").cast("double"))
    .withColumn("local_bid_price_d", F.col("display_price").cast("double"))
    .withColumn("bid_value_local_d", F.col("settled_market_value_pc").cast("double"))
    .withColumn("local_currency", F.upper(F.trim(F.col("position_currency").cast("string"))))
    .withColumn("report_date", F.to_date(F.col("position_as_of_date")))
    .withColumn("abc_value_d", F.col("settled_market_value_abc").cast("double"))
    .withColumn("account_base_currency_u", F.upper(F.trim(F.col("account_base_currency").cast("string"))))
    .withColumn("fx_rate_col_d", F.col("position_to_account_exchange_rate").cast("double"))
    .withColumn("raw_row_payload", F.col("raw_row_payload").cast("string"))
)

# join FX
d = (
    d
    .withColumn("ccy_u", F.col("local_currency"))
    .join(fx_df, F.col("ccy_u") == fx_df.currency_u, "left")
)

# GBP rules (from 11_dfm_wh_ireland.md):
# 1) Position Currency == GBP -> bid_value_gbp = PC, fx=1
# 2) else if Account Base Currency == GBP -> bid_value_gbp = ABC
# 3) else if Position to Account Exchange Rate present -> PC * fx_rate_col
# 4) else if fx_rates.csv available -> PC * rate_to_gbp
# 5) else null + FX_NOT_AVAILABLE flag
d = (
    d
    .withColumn(
        "fx_rate_out",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .when(F.col("account_base_currency_u") == "GBP", F.lit(None).cast("double"))
         .when(F.col("fx_rate_col_d").isNotNull(), F.col("fx_rate_col_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "bid_value_gbp_d",
        F.when(F.col("local_currency") == "GBP", F.col("bid_value_local_d"))
         .when((F.col("account_base_currency_u") == "GBP") & F.col("abc_value_d").isNotNull(), F.col("abc_value_d"))
         .when(F.col("fx_rate_col_d").isNotNull(), F.col("bid_value_local_d") * F.col("fx_rate_col_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("bid_value_local_d") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# data_quality_flags
d = (
    d
    .withColumn("flag_cash_defaulted", F.lit("CASH_DEFAULTED"))
    .withColumn("flag_accrued_defaulted", F.lit("ACCRUED_DEFAULTED"))
    .withColumn("flag_fx_not_available", F.when(F.col("bid_value_gbp_d").isNull(), F.lit("FX_NOT_AVAILABLE")))
    .withColumn("flag_raw_row_preserved", F.when(F.col("raw_row_payload").isNotNull(), F.lit("RAW_ROW_PRESERVED_STAGE1")))
    .withColumn(
        "data_quality_flags",
        F.array_remove(F.array(F.col("flag_cash_defaulted"), F.col("flag_accrued_defaulted"), F.col("flag_fx_not_available"), F.col("flag_raw_row_preserved")), None)
    )
)

# Row-level parse/quality errors for required essentials
# Stage 1 accepts raw data as-is; missing ISIN/SEDOL is OK (Stage 2 will apply cash mapping)
bad = d.filter(
    F.col("policy_id").isNull() |
    F.col("report_date").isNull()
).select(
    "source_file",
    "source_row_id",
    F.lit("Required field missing: policy_id/report_date").alias("error_message"),
    F.coalesce(F.col("raw_row_payload"), F.concat_ws("|", F.col("policy_id"), F.col("position_as_of_date").cast("string"))).alias("raw_value")
)

bad_count = bad.count()
bad_logged_count = 0
if bad_count > 0:
    bad_agg = (
        bad.groupBy("source_file").count()
        .withColumn("source_row_id", F.lit("AGGREGATED"))
        .withColumn("error_message", F.concat(F.lit("Stage 1 preservation: retained "), F.col("count").cast("string"), F.lit(" rows with missing policy_id/report_date for Stage 2")))
        .withColumn("raw_value", F.lit(""))
        .select("source_file", "source_row_id", "error_message", "raw_value")
    )
    bad_logged_count = bad_agg.count()
    write_parse_errors(bad_agg)

good = d

print("[A7] Generating source_row_id hash (SHA256)...")
good = good.withColumn(
    "source_row_id_final",
    F.sha2(
        F.concat_ws("|",
            F.lit(period),
            F.lit(dfm_id),
            F.col("policy_id"),
            F.col("isin_out"),
            F.col("report_date").cast("string"),
            F.col("holding_d").cast("string"),
            F.col("bid_value_local_d").cast("string"),
            F.col("source_file"),
            F.col("source_row_id")
        ),
        256
    )
)

# ---------- A5 exact canonical projection ----------
canonical = (
    good
    .withColumn("period", F.lit(period))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("dfm_id", F.lit(dfm_id))
    .withColumn("dfm_name", F.lit(dfm_name))
    .withColumn("policy_id_type", F.lit("DFM"))
    .withColumn("other_security_id", F.lit(None).cast("string"))
    .withColumn("id_type", F.lit(None).cast("string"))
    .withColumn("asset_name", F.col("raw_row_payload").cast("string"))
    .withColumn("cash_value_gbp", F.lit(0.0))
    .withColumn("accrued_interest_gbp", F.lit(0.0))
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "period",
        "run_id",
        "dfm_id",
        "dfm_name",
        "source_file",
        "source_sheet",
        F.col("source_row_id_final").alias("source_row_id"),
        "policy_id",
        "policy_id_type",
        "dfm_policy_id",
        "security_id",
        F.col("isin_out").alias("isin"),
        "other_security_id",
        "id_type",
        "asset_name",
        dec("holding_d").alias("holding"),
        dec("local_bid_price_d").alias("local_bid_price"),
        "local_currency",
        dec("fx_rate_out").alias("fx_rate"),
        dec("cash_value_gbp").alias("cash_value_gbp"),
        dec("bid_value_gbp_d").alias("bid_value_gbp"),
        dec("accrued_interest_gbp").alias("accrued_interest_gbp"),
        "report_date",
        "ingested_at",
        "data_quality_flags"
    )
)

print("[A8] Writing to canonical_holdings table...")
# Append mode (as per your request: speed first). If you want MERGE later, add de-dupe by source_row_id.
canonical.write.mode("append").saveAsTable("canonical_holdings")

rows_ingested = canonical.count()

print("[A9] Writing audit log...")
status = "PARTIAL" if bad_logged_count > 0 else "OK"
write_audit(
    files_processed=len(xlsx_files),
    rows_ingested=rows_ingested,
    parse_errors_count=bad_logged_count + len(parse_err_rows),
    drift_events_count=0,
    status=status
)

# ---------- A10: Exit ----------

mssparkutils.notebook.exit(status)